# 02 — Preprocessing

Missing categorical cells stay missing until the model pipeline imputes them (median for numbers, most frequent for categories). Numeric inputs are capped with a 3×IQR fence learned on the training split. `src/features.py` then adds Engagement_Index, Sleep_Deficit, Prior_x_Effort, and Motivation_Num.

Six columns stay out of the model: Gender, Family_Income, Parental_Education_Level, Learning_Disabilities, School_Type, Distance_from_Home. The accuracy cost of that choice is in `reports/feature_ablation.csv`.


In [1]:
from pathlib import Path
import pandas as pd
from src.train import DROPPED, load_dataset, prepare_frames, split_dataset

frame = load_dataset(Path("data/StudentPerformanceFactors.csv"))
print("Missing values by column:")
print(frame.isna().sum().to_string())
train, test = split_dataset(frame)
caps, train_prepared, test_prepared = prepare_frames(train, test)
print("Train/test:", len(train_prepared), len(test_prepared))
print("Caps:", caps)
print("Dropped:", DROPPED)
print("Engineered columns present:", all(col in train_prepared.columns for col in ["Engagement_Index", "Sleep_Deficit", "Prior_x_Effort", "Motivation_Num"]))


Missing values by column:
Hours_Studied                  0
Attendance                     0
Parental_Involvement           0
Access_to_Resources            0
Extracurricular_Activities     0
Sleep_Hours                    0
Previous_Scores                0
Motivation_Level               0
Internet_Access                0
Tutoring_Sessions              0
Family_Income                  0
Teacher_Quality               78
School_Type                    0
Peer_Influence                 0
Physical_Activity              0
Learning_Disabilities          0
Parental_Education_Level      90
Distance_from_Home            67
Gender                         0
Exam_Score                     0
Train/test: 5285 1322
Caps: {'Hours_Studied': (-8.0, 48.0), 'Attendance': (10.0, 150.0), 'Sleep_Hours': (0.0, 14.0), 'Previous_Scores': (-12.0, 163.0), 'Tutoring_Sessions': (-2.0, 5.0), 'Physical_Activity': (-4.0, 10.0)}
Dropped: ['Gender', 'Family_Income', 'Parental_Education_Level', 'Learning_Disabilities', 'Sc

In [2]:
ablation = pd.read_csv("reports/feature_ablation.csv")
print(ablation.to_string(index=False))


                       Feature set  Typ_RMSE   Typ_R2  RMSE_change_vs_compact_%
                   All 23 features  0.344542 0.989373                -57.670394
        Compact 17 features (used)  0.813951 0.940693                  0.000000
                  Compact + Gender  0.813940 0.940694                 -0.001352
           Compact + Family_Income  0.710253 0.954842                -12.739999
Compact + Parental_Education_Level  0.703581 0.955686                -13.559766
   Compact + Learning_Disabilities  0.746697 0.950089                 -8.262649
             Compact + School_Type  0.814072 0.940675                  0.014863
      Compact + Distance_from_Home  0.735333 0.951596                 -9.658735
